# Laboratorio de regresión - 4

|                |   |
:----------------|---|
| **Nombre**      Valeria Estefanía Milke Loera|   |
| **Fecha**      07-02-2026 |   |
| **Expediente** 739228 |   |

## Modelos penalizados

Hasta ahora la función de costo que usamos para decidir qué tan bueno es nuestro modelo al momento de ajustar es:

$$ \text{RSS} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y_i})^2 $$

Dado que los errores obtenidos son una combinación de sesgo y varianza, puede ser que se sesgue un parámetro para minimizar el error. Esto significa que el modelo puede decidir que la salida no sea una combinación de los factores, sino una fuerte predilección sobre uno de los factores solamente. 

E.g. se quiere ajustar un modelo

$$ \hat{z} = \hat{\beta_0} + \hat{\beta_1} x + \hat{\beta_2} y $$

Se ajusta el modelo y se decide que la mejor decisión es $\hat{\beta_1} = 10000$ y $\hat{\beta_2}=50$. Considera limitaciones de problemas reales:
- Quizás los parámetros son ajustes de maquinaria que se deben realizar para conseguir el mejor producto posible, y que $10000$ sea imposible de asignar.
- Quizás los datos actuales están sesgados y sólo hacen parecer que uno de los factores importa más que el otro.

Una de las formas en las que se puede mitigar este problema es penalizando a los parámetros del modelo, cambiando la función de costo:

$$ \text{RSS}_{L2} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p \hat{\beta_j}^2 $$

El *L2* significa que se está agregando una penalización de segundo orden. Lo que hace esta penalización es que los factores ahora sólo tendrán permitido crecer si hay una reducción al menos proporcional en el error (sacrificamos sesgo, pero reducimos la varianza).

Asimismo, existe la penalización *L1*

$$ \text{RSS}_{L1} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p |\hat{\beta_j}| $$

A las penalizaciones *L2* y *L1* se les conoce también como Ridge y Lasso, respectivamente.

Para realizar una regresión con penalización de Ridge o de Lasso usamos el objeto `Ridge(alpha=?)` o `Lasso(alpha=?)` en lugar de `LinearRegression()` de `sklearn`.

Utiliza el dataset de publicidad (Advertising.csv) y realiza 3 regresiones múltiples:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

1. Sin penalización
2. Con penalización L2
3. Con penalización L1

¿Qué puedes observar al ajustar los valores de `alpha`? 

Compara los resultados de los coeficientes utilizando valores diferentes de $\alpha$ y los $R^2$ resultantes.



In [26]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

In [10]:
df = pd.read_csv(r"C:\Users\valer\Advertising.csv")

**Regresión múltiple sin penalización**

In [13]:
x = df[["TV","radio","newspaper"]]
y = df[["sales"]]

In [14]:
lr = LinearRegression()
lr.fit(x,y)

LinearRegression()

In [16]:
lr.intercept_

array([2.93888937])

In [18]:
lr.coef_

array([[ 0.04576465,  0.18853002, -0.00103749]])

Modelo resultante

y = 2.9389 + 0.0458(TV) + 0.1885(radio) - 0.0010(newspaper)

**Cálculo R^2**

In [19]:
r2 = lr.score(x,y)
print("El estadístico R2 es:",r2)

El estadístico R2 es: 0.8972106381789522


___________________________

**Escalamiento MinMax**

In [21]:
dfscaled = df.copy()
for col in dfscaled.columns: 
    minn = dfscaled[col].min()
    dfscaled[col] = (dfscaled[col]-minn)
    maxx = dfscaled[col].max()
    dfscaled[col]/=maxx

In [61]:
dfscaled.describe()

,Unnamed: 0,TV,radio,newspaper,sales
count,200.00000,200.000000,200.000000,200.000000,200.000000
mean,0.50000,0.494902,0.469032,0.266086,0.489075
std,0.29085,0.290342,0.299331,0.191545,0.205412
min,0.00000,0.000000,0.000000,0.000000,0.000000
25%,0.25000,0.249155,0.201109,0.109499,0.345472
50%,0.50000,0.504058,0.461694,0.223835,0.444882
75%,0.75000,0.737656,0.736391,0.394019,0.622047
max,1.00000,1.000000,1.000000,1.000000,1.000000


_____________

### Regresión múltiple con penalización L2

In [52]:
x2 = dfscaled[["TV","radio","newspaper"]]
y2 = df[["sales"]]

In [76]:
lr2 = Ridge()
lr2.fit(x2,y2)

Ridge()

In [77]:
lr2.intercept_

array([3.50018391])

In [78]:
lr2.coef_

array([12.78795999,  8.82156031,  0.21020893])

**Modelo Resultante**

y = 3.50018391 + 12.78795999(TV) + 8.82156031(radio) + 0.21020893(newspaper)

**Cálculo R^2**

In [85]:
r22 = lr2.score(x2,y2)
print("El estadístico R2 es:",r22)

El estadístico R2 es: 0.8946032181214243


**Comparación de modelos Ridge para distintos valores de α**

In [102]:
alphas = [0.01, 0.1, 0.5, 2,3]
for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(x2,y2)
    i = ridge.intercept_
    c = ridge.coef_
    r2 = ridge.score(x2,y2)
    print("Alpha",a)
    print("Modelo resultante:",i,"+",c[0],"(TV) +",c[1],"(radio) +",c[2],"(newspaper)")
    print("El estadístico R2 es:",r2)
    print("______________________________")

Alpha 0.01
Modelo resultante: [2.97613905] + 13.524717550551314 (TV) + 9.345359656386623 (radio) + -0.1139592185789286 (newspaper)
El estadístico R2 es: 0.8972103415616504
______________________________
Alpha 0.1
Modelo resultante: [3.02565778] + 13.454148994355611 (TV) + 9.294233664734593 (radio) + -0.07868673613309911 (newspaper)
El estadístico R2 es: 0.8971813407837388
______________________________
Alpha 0.5
Modelo resultante: [3.24115137] + 13.149467091142945 (TV) + 9.075970222474092 (radio) + 0.06287141604076175 (newspaper)
El estadístico R2 es: 0.8965162076376907
______________________________
Alpha 2
Modelo resultante: [3.98547701] + 12.122601087443039 (TV) + 8.363433054750327 (radio) + 0.43145588887335046 (newspaper)
El estadístico R2 es: 0.8879378390832228
______________________________
Alpha 3
Modelo resultante: [4.43043369] + 11.524100796232235 (TV) + 7.9594932623578565 (radio) + 0.5844253530388631 (newspaper)
El estadístico R2 es: 0.8784855950868729
_______________________

**Interpretación**

Al analizar distintos valores del parámetro α en la regresión Ridge, se observa que al aumentar la penalización los coeficientes del modelo se reducen progresivamente. Al mismo tiempo, el coeficiente R^2 presenta una disminución gradual, lo que indica que el modelo explica ligeramente menos variabilidad de la variable respuesta. Sin embargo, esta reducción en el R^2 no es drástica, lo que sugiere que la penalización L2 permite controlar la magnitud de los coeficientes sin afectar de forma significativa la capacidad explicativa del modelo.

______________

### Regresión múltiple con penalización L1

In [80]:
x1 = dfscaled[["TV","radio","newspaper"]]
y1 = df[["sales"]]

In [81]:
lr1 = Lasso()
lr1.fit(x1,y1)

Lasso()

In [82]:
lr1.intercept_

array([12.96618497])

In [83]:
lr1.coef_

array([2.13439264, 0.        , 0.        ])

**Modelo resultante**

y = 12.96618497 + 2.13439264(TV)

**Cálculo R^2**

In [95]:
r21 = lr1.score(x1,y1)
print("El estadístico R2 es:",r21)

El estadístico R2 es: 0.1717102154785033


**Comparación de modelos Lasso para distintos valores de α**

In [103]:
alphas = [0.01, 0.1, 0.5, 2,3]
for a in alphas:
    lasso = Lasso(alpha=a)
    lasso.fit(x1,y1)
    i = lasso.intercept_
    c = lasso.coef_
    r2 = lasso.score(x1,y1)
    print("Alpha",a)
    print("Modelo resultante:",i,"+",c[0],"(TV) +",c[1],"(radio) +",c[2],"(newspaper)")
    print("El estadístico R2 es:",r2)
    print("______________________________")

Alpha 0.01
Modelo resultante: [3.05895076] + 13.416474972634484 (TV) + 9.218363639548233 (radio) + -0.0 (newspaper)
El estadístico R2 es: 0.8971132687194486
______________________________
Alpha 0.1
Modelo resultante: [4.01135312] + 12.39746016930619 (TV) + 8.263013639010145 (radio) + 0.0 (newspaper)
El estadístico R2 es: 0.8890950238394831
______________________________
Alpha 0.5
Modelo resultante: [8.24425153] + 7.868507529979379 (TV) + 4.017013528053099 (radio) + 0.0 (newspaper)
El estadístico R2 es: 0.6947133993873196
______________________________
Alpha 2
Modelo resultante: [14.0225] + 0.0 (TV) + 0.0 (radio) + 0.0 (newspaper)
El estadístico R2 es: 0.0
______________________________
Alpha 3
Modelo resultante: [14.0225] + 0.0 (TV) + 0.0 (radio) + 0.0 (newspaper)
El estadístico R2 es: 0.0
______________________________


**Interpretación**

Al analizar la regresión Lasso para distintos valores del parámetro α, se observa que al incrementar la penalización algunos coeficientes se hacen exactamente cero, lo cual indica que el modelo realiza una selección automática de variables. Conforme aumenta α, el número de variables incluidas en el modelo disminuye y el coeficiente R^2 se reduce de manera significativa. Para valores elevados de α, el modelo queda compuesto únicamente por el intercepto, lo que provoca un R^2 cercano a cero y refleja un fuerte subajuste.